### ** Create Table for Bronze, Silver, Gold **
parameterized environment..

Existing tables (rawCustomer, rawDiscount, rawSales, rawInvoice) kept as-is. New entities (rawRegion, rawProduct, rawStore) point to AWS S3.

Gold Layer — Dimensions, Facts, Date
Existing `refinedSales`, `refinedCustomer`, `refinedInvoice` kept.
All new dims and facts **point to AWS S3** and use surrogate identity keys.

In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev", choices=["dev", "qa", "prd"], label = "Select Environment")
env = dbutils.widgets.get("environment")
print(env)

In [0]:
%sql
-- ===== 1. EXISTING: Customer (managed table) =====
CREATE OR REPLACE TABLE saleslake_${environment}.bronze_schema_${environment}.rawCustomer
(
    customer_id string , customer_name string, email string, 
    phone string , address string , 
    city string , state string, 
    country string,zip_code string,
    segment string,
    ingest_ts timestamp
);

CREATE OR REPLACE TABLE saleslake_${environment}.silver_schema_${environment}.cleanedcustomer
(
    customer_id int , customer_name string, email string, 
    phone string , address string , 
    city string , state string, 
    country string,zip_code string,
    segment string,
    ingest_ts timestamp
);

CREATE OR REPLACE TABLE saleslake_${environment}.gold_schema_${environment}.refinedcustomer
(
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id int , customer_name string, email string, 
    phone string , address string , 
    city string , state string, 
    country string,zip_code string,
    segment string,
    ingest_ts timestamp,
    is_active string,
    rec_version	int, 
    start_effective_ts	timestamp,
    end_effective_ts timestamp

);

In [0]:
%sql
-- ===== 2. EXISTING: Sales  (managed table) =====
CREATE OR REPLACE TABLE saleslake_${environment}.bronze_schema_${environment}.rawSales
(
    sale_id string , product string, category string, 
    quantity string , price string , 
    sale_date string , region string, 
    ingest_ts timestamp
);

CREATE OR REPLACE TABLE saleslake_${environment}.silver_schema_${environment}.cleanedSales
(
    sale_id INTEGER , product string, category string, 
    quantity INTEGER , price double , 
    sale_date date , region string, 
    ingest_ts timestamp
);

-- ===== EXISTING: Sales  (SCD1, existing schema) =====
CREATE OR REPLACE TABLE saleslake_${environment}.gold_schema_${environment}.refinedSales 
(
    sale_id INT, product STRING, category STRING,
    quantity INT, price DOUBLE, sale_date DATE, region STRING,
    initial_load_ts TIMESTAMP, last_updt_ts TIMESTAMP
);